In [ ]:
import os
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
bundle = next(
    (candidate for path in (cwd, *cwd.parents)
     for candidate in (path, path / "final_code")
     if (candidate / "finetuning" / "finetuning.ipynb").is_file()),
    None,
)
if bundle is None:
    raise FileNotFoundError("Open this notebook from within the project directory.")

notebook_dir = bundle / "finetuning"
os.chdir(notebook_dir)
sys.path.insert(0, str(notebook_dir))

In [ ]:
pwd

### Chronos_t5_large

In [ ]:
# !pip install "chronos-forecasting[dev] @ git+https://github.com/amazon-science/chronos-forecasting.git"
# !git clone https://github.com/amazon-science/chronos-forecasting.git
# !pip install "chronos-forecasting[training] @ git+https://github.com/amazon-science/chronos-forecasting.git"
# cd chronos-forecasting
# !pip install -e ".[training]"
# !pip install typer-config

In [ ]:
import sys
sys.path.insert(0, '../dependencies/chronos-forecasting')

import json
import numpy as np
import pandas as pd
import torch
from pprint import pprint
from pathlib import Path
from transformers import Trainer, TrainingArguments, set_seed, AutoModelForSeq2SeqLM, EarlyStoppingCallback
from gluonts.dataset.arrow import ArrowWriter
from gluonts.dataset.common import FileDataset
from chronos import ChronosConfig, ChronosPipeline
from scripts.training.train import ChronosDataset 

Data Load

In [ ]:
# Lab Data
normal_df = pd.read_csv('../data/2st_collection/normal/normal_B-M-S1_imu_data_60Hz.csv', header=1)

features = [
    "accel_x","accel_y","accel_z",
    "gyro_x","gyro_y","gyro_z",
    "mag_x","mag_y","mag_z",
]

In [ ]:
# Ds Data
ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')

date = "2025-11-10"
ds_press_imu["timestamp"] = pd.to_datetime(ds_press_imu["timestamp"], utc=True, errors="coerce")
ds_press_imu = ds_press_imu.dropna(subset=["timestamp"])

mask = ds_press_imu["timestamp"].dt.date == pd.Timestamp(date).date()
normal_df = ds_press_imu[mask].reset_index(drop=True)

features = [
    "acc_x","acc_y","acc_z",
    "gyro_x","gyro_y","gyro_z",
    # "mag_x","mag_y","mag_z",
]

In [ ]:
arr = normal_df[features].to_numpy().astype(np.float32)

split_idx = int(len(arr) * 0.9)
train_arr = arr[:split_idx]
val_arr = arr[split_idx:]

Tokenization

In [ ]:
# Arrorw format
def convert_to_arrow(path, time_series, compression="lz4"):
    start = np.datetime64("2000-01-01 00:00", "s")
    dataset = [{"start": start, "target": ts} for ts in time_series]
    ArrowWriter(compression=compression).write_to_file(dataset, path=path)

# Train
train_series = [train_arr[:, i] for i in range(train_arr.shape[1])]
train_path = Path("./train_data.arrow")
convert_to_arrow(train_path, train_series)

# Val
val_series = [val_arr[:, i] for i in range(val_arr.shape[1])]
val_path = Path("./val_data.arrow")
convert_to_arrow(val_path, val_series)

set_seed(42)

In [ ]:
# 공식 pretrained 모델의 config
from chronos import ChronosPipeline

checking_pipeline = ChronosPipeline.from_pretrained("amazon/chronos-t5-large")
pprint(checking_pipeline.model.config)

In [ ]:
model_id = "amazon/chronos-t5-large"
vocab_size = 4096

model = AutoModelForSeq2SeqLM.from_pretrained(model_id) # 모델 가중치 로드
model.resize_token_embeddings(vocab_size)

# 공식문서 참고
chronos_config = ChronosConfig(
    tokenizer_class="MeanScaleUniformBins",
    tokenizer_kwargs={"low_limit": -15.0, "high_limit": 15.0},
    context_length=512,
    prediction_length=64,
    n_tokens=vocab_size,
    n_special_tokens=2,
    pad_token_id=0,
    eos_token_id=1,
    use_eos_token=True,
    model_type="seq2seq",
    num_samples=20,
    temperature=1.0,
    top_k=50,
    top_p=1.0,
)

tokenizer = chronos_config.create_tokenizer()

In [ ]:
# ChronosDataset 기본값 확인
from scripts.training.train import ChronosDataset
import inspect

sig = inspect.signature(ChronosDataset.__init__)
for param_name, param in sig.parameters.items():
    if param.default is not inspect.Parameter.empty:
        print(f"{param_name}: {param.default}")

In [ ]:
train_dataset = ChronosDataset(
    datasets=[FileDataset(train_path, freq="S")],
    probabilities=[1.0],
    tokenizer=tokenizer,
    context_length=512,
    prediction_length=64,
    min_past=None,
    mode="training",
)

val_dataset = ChronosDataset(
    datasets=[FileDataset(val_path, freq="S")],
    probabilities=[1.0],
    tokenizer=tokenizer,
    context_length=512,
    prediction_length=64,
    min_past=None,
    mode="validation",
)

Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./weights/chronos-t5-large/chronos-t5-large_ds_full_finetuned/",
    
    # 논문 참고
    max_steps=100_000,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=1e-3,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ratio=0.0,
    optim="adamw_torch_fused",
    
    # Validation
    eval_strategy="steps",       # 스텝 단위로 평가
    eval_steps=1000,             # 1000 스텝마다 validation
    
    # Early Stopping
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_steps=1000,
    
    # etc
    gradient_accumulation_steps=2,
    logging_steps=500,
    save_total_limit=3,          # 최근 3개 체크포인트만 유지
    save_strategy="steps",       # eval_strategy와 동일해야 함
    # dataloader_num_workers=1,
    dataloader_num_workers=0,
    tf32=True, # GPU
    torch_compile=False,
    report_to="none",
    remove_unused_columns=False,
)

early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=5,      # 5번 연속 개선 없으면 중단
    early_stopping_threshold=0.001,  # 최소 개선 폭 (optional)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,    # validation 데이터셋 추가
    callbacks=[early_stopping_callback],
)

trainer.train()
trainer.save_model()

save_path = Path("./weights/chronos-t5-large/chronos-t5-large_ds_full_finetuned/")
save_path.mkdir(parents=True, exist_ok=True)

with open(save_path / "chronos_config.json", "w") as f:
    json.dump(vars(chronos_config), f, indent=2)

# chronos_config.save_pretrained(save_path)

print("Complete Fine-tuning")

###### TMP

In [ ]:
import sys
sys.path.insert(0, '../dependencies/chronos-forecasting')

from pprint import pprint
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from transformers import Trainer, TrainingArguments, set_seed, AutoModelForSeq2SeqLM, AutoConfig
from gluonts.dataset.arrow import ArrowWriter
from gluonts.dataset.common import FileDataset
from chronos import ChronosConfig, ChronosPipeline, ChronosTokenizer
from scripts.training.train import ChronosDataset

In [ ]:
normal_df = pd.read_csv('../data/2st_collection/normal/normal_B-M-S1_imu_data_60Hz.csv', header=1)
features = [
    "accel_x","accel_y","accel_z",
    "gyro_x","gyro_y","gyro_z",
    "mag_x","mag_y","mag_z",
]
arr = normal_df[features].to_numpy().astype(np.float32)

In [ ]:
# Arror format
def convert_to_arrow(path, time_series, compression="lz4"):
    start = np.datetime64("2000-01-01 00:00", "s")
    dataset = [{"start": start, "target": ts} for ts in time_series]
    ArrowWriter(compression=compression).write_to_file(dataset, path=path)

time_series_list = [arr[:, i] for i in range(arr.shape[1])]
data_path = Path("./imu_data.arrow")
convert_to_arrow(data_path, time_series_list)

set_seed(42)

In [ ]:
# 공식 pretrained 모델의 config
from chronos import ChronosPipeline

checking_pipeline = ChronosPipeline.from_pretrained("amazon/chronos-t5-large")
pprint(checking_pipeline.model.config)

In [ ]:
model_id = "amazon/chronos-t5-large"
vocab_size = 4096 # paper 설정

# 모델 로드
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)
model.resize_token_embeddings(vocab_size)

# Chronos Config 생성 (논문 설정)
chronos_config = ChronosConfig(
    tokenizer_class="MeanScaleUniformBins",
    tokenizer_kwargs={"low_limit": -15.0, "high_limit": 15.0},
    context_length=512,
    prediction_length=64,
    n_tokens=vocab_size,
    n_special_tokens=2,
    pad_token_id=0,
    eos_token_id=1,
    use_eos_token=True,
    model_type="seq2seq",
    
    num_samples=20,
    temperature=1.0,
    top_k=50,
    top_p=1.0,
)
tokenizer = chronos_config.create_tokenizer()

In [ ]:
# Dataset
train_dataset = ChronosDataset(
    datasets=[FileDataset(data_path, freq="S")],
    probabilities=[1.0],
    tokenizer=tokenizer,
    context_length=512,
    prediction_length=64,
    min_past=60,
    mode="training",
)

In [ ]:
# print(vars(checking_pipeline.config))
from scripts.training.train import ChronosDataset
import inspect
pprint(inspect.signature(ChronosDataset.__init__))

In [ ]:
# Training
training_args = TrainingArguments(
    output_dir="./weights/chronos-t5-large/chronos-t5-large_full_finetuned/",
    
    max_steps=200_000,
    per_device_train_batch_size=32,
    learning_rate=1e-3,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_ratio=0.0,
    optim="adamw_torch_fused",
    
    gradient_accumulation_steps=2,
    logging_steps=500,
    save_steps=50_000,
    dataloader_num_workers=1,
    tf32=False, #True,
    torch_compile=False,
    report_to="none",
    remove_unused_columns=False,
)

# Trainer 실행
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

print("학습 시작")
trainer.train()
trainer.save_model()

# Config 저장
chronos_config.save_pretrained("./weights/chronos-t5-large/chronos-t5-large_full_finetuned/")

print("Fine-tuning 완료!")

In [ ]:
# Inference
ckpt_path = "./weights/chronos-t5-large/chronos-t5-large_full_finetuned/"

pipeline = ChronosPipeline.from_pretrained(
    "./weights/chronos-t5-large/chronos-t5-large_full_finetuned/",
    device_map="cuda",
    torch_dtype=torch.bfloat16,
)

context = torch.tensor(arr[-512:, 0])
forecast = pipeline.predict(context, prediction_length=64)
print(f"Forecast shape: {forecast.shape}")

### Chronos 2

In [ ]:
from chronos import Chronos2Pipeline, BaseChronosPipeline
from tqdm import tqdm

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch
import os

import warnings
warnings.filterwarnings('ignore')

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
pipeline: Chronos2Pipeline = BaseChronosPipeline.from_pretrained("amazon/chronos-2", device_map="cuda" ) #cuda

In [ ]:
# Lab Data
normal_df = pd.read_csv('../data/2st_collection/normal/normal_B-M-S1_imu_data_60Hz.csv', header=1)

features = [
    "accel_x","accel_y","accel_z",
    "gyro_x","gyro_y","gyro_z",
    "mag_x","mag_y","mag_z",
]

In [ ]:
# Ds Data

ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')

date = "2025-11-10"
ds_press_imu["timestamp"] = pd.to_datetime(ds_press_imu["timestamp"], utc=True, errors="coerce")
ds_press_imu = ds_press_imu.dropna(subset=["timestamp"])

mask = ds_press_imu["timestamp"].dt.date == pd.Timestamp(date).date()
normal_df = ds_press_imu[mask].reset_index(drop=True)

features = [
    "acc_x","acc_y","acc_z",
    "gyro_x","gyro_y","gyro_z",
    # "mag_x","mag_y","mag_z",
]

In [ ]:
arr = normal_df[features].to_numpy().astype(np.float32) # (T, 9)
T = arr.shape[0]
target = arr.T 

prediction_length = 64
learning_rate = 1e-6
num_steps = 50000
batch_size = 16 #256

# train_inputs = [{"target": target,}]
# if Validation
C = target.shape[1]
val_size = int(C * 0.1)
train_size = C - val_size

target_train = target[:, :train_size]
target_val   = target[:, train_size:]

train_inputs = [{"target": target_train,}]
validation_inputs = [{"target": target_val,}]

공식 문서 참고


https://github.com/amazon-science/chronos-forecasting/blob/main/notebooks/chronos-2-quickstart.ipynb

In [ ]:
finetuned_pipeline = pipeline.fit(
    inputs=train_inputs,
    validation_inputs=validation_inputs,
    prediction_length=prediction_length,
    num_steps=num_steps,
    learning_rate=learning_rate,
    batch_size=batch_size,
    logging_steps=100,
    finetune_mode="lora",#loar
)

# save_full_finetuned_dir = "./weights/chronos2/chronos2_full_finetuned/"
save_full_finetuned_dir = "./weights/chronos2/chronos2_ds_lora_finetuned/"
# save_full_finetuned_dir = "./weights/chronos2/chronos2_lora_finetuned/"
finetuned_pipeline.save_pretrained(save_full_finetuned_dir)
print("Saved to:", save_full_finetuned_dir)

### TimesFM

https://github.com/google-research/timesfm/blob/master/src/timesfm/timesfm_2p5/timesfm_2p5_torch.py

In [ ]:
from torch.utils.data import Dataset, DataLoader
from timesfm.torch import util
from tqdm import tqdm

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import timesfm

revin = util.revin

In [ ]:
# Lab Data
normal_df = pd.read_csv('../data/2st_collection/normal/normal_B-M-S1_imu_data_60Hz.csv', header=1)
features = ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z", "mag_x", "mag_y", "mag_z"]

In [ ]:
# Ds Data
ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')

date = "2025-11-10"
ds_press_imu["timestamp"] = pd.to_datetime(ds_press_imu["timestamp"], utc=True, errors="coerce")
ds_press_imu = ds_press_imu.dropna(subset=["timestamp"])

mask = ds_press_imu["timestamp"].dt.date == pd.Timestamp(date).date()
normal_df = ds_press_imu[mask].reset_index(drop=True)

features = [
    "acc_x","acc_y","acc_z",
    "gyro_x","gyro_y","gyro_z",
    # "mag_x","mag_y","mag_z",
]

In [ ]:
arr = normal_df[features].to_numpy().astype(np.float32)

In [ ]:
class TimesFMDataset(Dataset):
    def __init__(self, data, context_len=512, horizon_len=128, patch_len=32):
        self.data = data
        self.context_len = context_len
        self.horizon_len = horizon_len
        self.patch_len = patch_len
        self.num_features = data.shape[1]
        self.num_patches = context_len // patch_len
        self.total_len = context_len + horizon_len
        self.num_windows = max(0, len(data) - self.total_len + 1)

    def __len__(self):
        return self.num_windows * self.num_features

    def __getitem__(self, idx):
        feature_idx = idx % self.num_features
        window_idx = idx // self.num_features
        ts = self.data[window_idx:window_idx + self.total_len, feature_idx]

        context = ts[:self.context_len]
        future = ts[self.context_len:]

        context_patches = context.reshape(self.num_patches, self.patch_len)
        padding_patches = np.zeros_like(context_patches, dtype=bool)

        return (
            torch.tensor(context_patches, dtype=torch.float32),
            torch.tensor(padding_patches, dtype=torch.bool),
            torch.tensor(future, dtype=torch.float32),
        )

In [ ]:
def compute_patch_stats(patches, masks):
    """패치별 running statistics 계산 (공식 코드 방식)"""
    B, num_patches, p = patches.shape
    device = patches.device
    
    n = torch.zeros(B, device=device)
    mu = torch.zeros(B, device=device)
    sigma = torch.ones(B, device=device)  # 초기값 1
    
    patch_mu, patch_sigma = [], []
    
    for i in range(num_patches):
        patch_data = patches[:, i, :]   # (B, p)
        patch_mask = masks[:, i, :]     # (B, p)
        
        # 유효한 데이터만 사용
        valid = ~patch_mask
        valid_count = valid.sum(dim=1).float().clamp(min=1)
        
        # Mean 업데이트
        patch_sum = (patch_data * valid).sum(dim=1)
        new_n = n + valid_count
        new_mu = (n * mu + patch_sum) / new_n.clamp(min=1)
        
        # Std 업데이트 (simplified)
        patch_sq = ((patch_data - new_mu.unsqueeze(-1)) ** 2 * valid).sum(dim=1)
        new_var = (n * sigma**2 + patch_sq) / new_n.clamp(min=1)
        new_sigma = torch.sqrt(new_var + 1e-8)
        
        n, mu, sigma = new_n, new_mu, new_sigma
        patch_mu.append(mu.clone())
        patch_sigma.append(sigma.clone())
    
    return torch.stack(patch_mu, dim=1), torch.stack(patch_sigma, dim=1)  # (B, num_patches)

In [ ]:
def finetune_timesfm(model, train_loader, val_loader=None, num_epochs=10, lr=1e-5,
                     device='cuda' if torch.cuda.is_available() else 'cpu',
                     save_path='./weights/timesfm/timesfm_ds_finetuned.pth'):

    tfm_model = model.model
    tfm_model.to(device)
    
    o = tfm_model.o  # 128
    q = tfm_model.q  # 10

    for param in tfm_model.parameters():
        param.requires_grad = True

    print(f"Device: {device}")
    print(f"Trainable params: {sum(p.numel() for p in tfm_model.parameters() if p.requires_grad):,}")

    optimizer = torch.optim.AdamW(tfm_model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    loss_fn = nn.MSELoss()
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        tfm_model.train()
        train_loss = 0.0

        for context_patches, padding_patches, x_future in tqdm(train_loader):
            context_patches = context_patches.to(device)
            padding_patches = padding_patches.to(device)
            x_future = x_future.to(device)
            
            B, num_patches, p = context_patches.shape
            target_len = x_future.shape[-1]

            # 1. Running stats 계산
            context_mu, context_sigma = compute_patch_stats(context_patches, padding_patches)
            
            # 2. RevIN 정규화
            normed_inputs = revin(context_patches, context_mu, context_sigma, reverse=False)
            normed_inputs = torch.where(padding_patches, torch.zeros_like(normed_inputs), normed_inputs)

            # 3. Forward
            optimizer.zero_grad()
            output_tuple, _ = tfm_model(normed_inputs, padding_patches)
            
            pred_raw = output_tuple[2]  # (B, num_patches, o*q)
            normed_preds = pred_raw.reshape(B, num_patches, o, q)
            
            # 4. RevIN 역정규화
            renormed_preds = revin(normed_preds, context_mu, context_sigma, reverse=True)
            
            # 5. 예측 추출 (index 5 = median, 공식 코드 기준)
            predictions_mean = renormed_preds[..., 5]  # (B, num_patches, o)
            last_patch_pred = predictions_mean[:, -1, :target_len]

            # 6. Loss & Backward
            loss = loss_fn(last_patch_pred, x_future)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tfm_model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        scheduler.step()

        # Validation
        if val_loader:
            tfm_model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for context_patches, padding_patches, x_future in tqdm(val_loader):
                    context_patches = context_patches.to(device)
                    padding_patches = padding_patches.to(device)
                    x_future = x_future.to(device)
                    
                    B, num_patches, p = context_patches.shape
                    target_len = x_future.shape[-1]
                    
                    context_mu, context_sigma = compute_patch_stats(context_patches, padding_patches)
                    normed_inputs = revin(context_patches, context_mu, context_sigma, reverse=False)
                    normed_inputs = torch.where(padding_patches, torch.zeros_like(normed_inputs), normed_inputs)
                    
                    output_tuple, _ = tfm_model(normed_inputs, padding_patches)
                    pred_raw = output_tuple[2]
                    normed_preds = pred_raw.reshape(B, num_patches, o, q)
                    renormed_preds = revin(normed_preds, context_mu, context_sigma, reverse=True)
                    
                    predictions_mean = renormed_preds[..., 5]
                    last_patch_pred = predictions_mean[:, -1, :target_len]
                    val_loss += loss_fn(last_patch_pred, x_future).item()

            val_loss /= len(val_loader)
            print(f"Epoch {epoch+1}/{num_epochs} | Train: {train_loss:.6f} | Val: {val_loss:.6f}")
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(tfm_model.state_dict(), save_path)
                print(f"  -> Saved best model")
        else:
            print(f"Epoch {epoch+1}/{num_epochs} | Train: {train_loss:.6f}")

    return tfm_model

In [ ]:
# Training
CONTEXT_LEN = 512
HORIZON_LEN = 64
BATCH_SIZE = 128
NUM_EPOCHS = 10
LR = 1e-5

torch.set_float32_matmul_precision("high")
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
PATCH_LEN = model.model.p

train_size = int(len(arr) * 0.9)
train_dataset = TimesFMDataset(arr[:train_size], CONTEXT_LEN, HORIZON_LEN, PATCH_LEN)
val_dataset = TimesFMDataset(arr[train_size:], CONTEXT_LEN, HORIZON_LEN, PATCH_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

save_path = Path("./weights/timesfm/timesfm_ds_full_finetuned/timesfm_full_finetuned.safetensors")
save_path.parent.mkdir(parents=True, exist_ok=True)

finetuned = finetune_timesfm(
    model, train_loader, val_loader, NUM_EPOCHS, LR,
    save_path=save_path
)
print("Complete Fine-tuning")

###### TMP

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import timesfm
from timesfm.torch import util

normal_df = pd.read_csv('../data/2st_collection/normal/normal_B-M-S1_imu_data_60Hz.csv', header=1)

features = ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z", "mag_x", "mag_y", "mag_z"]
arr = normal_df[features].to_numpy().astype(np.float32)

# Dataset
class TimesFMDataset(Dataset):
    def __init__(self, data, context_len=512, horizon_len=128, patch_len=32):
        self.data = data
        self.context_len = context_len
        self.horizon_len = horizon_len
        self.patch_len = patch_len
        self.num_features = data.shape[1]

        assert context_len % patch_len == 0
        self.num_patches = context_len // patch_len

        self.total_len = context_len + horizon_len
        self.num_windows = max(0, len(data) - self.total_len + 1)

    def __len__(self):
        return self.num_windows * self.num_features

    def __getitem__(self, idx):
        feature_idx = idx % self.num_features
        window_idx = idx // self.num_features
        ts = self.data[window_idx:window_idx + self.total_len, feature_idx]

        context = ts[:self.context_len]
        future = ts[self.context_len:]

        context_patches = context.reshape(self.num_patches, self.patch_len)
        padding_patches = np.zeros_like(context_patches, dtype=bool)

        return (
            torch.tensor(context_patches, dtype=torch.float32),
            torch.tensor(padding_patches, dtype=torch.bool),
            torch.tensor(future, dtype=torch.float32),
        )


# finetuning
def finetune_timesfm(model, train_loader, val_loader=None, num_epochs=10, lr=1e-5,
                     device='cuda' if torch.cuda.is_available() else 'cpu',
                     save_path='./weights/timesfm/timesfm_full_finetuned/timesfm_finetuned.pth'):

    tfm_model = model.model
    tfm_model.to(device)
    tfm_model.train()

    o = tfm_model.o
    q = tfm_model.q

    for param in tfm_model.parameters():
        param.requires_grad = True

    print(f"Device: {device}")
    print(f"Trainable params: {sum(p.numel() for p in tfm_model.parameters() if p.requires_grad):,}")

    optimizer = torch.optim.AdamW(tfm_model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    loss_fn = nn.MSELoss()
    best_val_loss = float('inf')

    for epoch in range(num_epochs):
        tfm_model.train()
        train_loss = 0.0

        for context_patches, padding_patches, x_future in train_loader:
            context_patches = context_patches.to(device)
            padding_patches = padding_patches.to(device).bool()
            x_future = x_future.to(device)

            optimizer.zero_grad()

            output_tuple, _ = tfm_model(context_patches, padding_patches)

            # output_tuple[2]: (B, num_patches, o*q) -> reshape -> (B, num_patches, o, q)
            pred_raw = output_tuple[2]
            B, num_patches, _ = pred_raw.shape
            predictions = pred_raw.reshape(B, num_patches, o, q)
            predictions_mean = predictions[..., 0]  # (B, num_patches, o)

            # 마지막 patch에서 horizon_len만큼 추출
            target_len = x_future.shape[-1]
            if target_len > o:
                raise ValueError(f"horizon_len ({target_len}) must be <= output_patch_len ({o})")
            last_patch_pred = predictions_mean[:, -1, :target_len]

            loss = loss_fn(last_patch_pred, x_future)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(tfm_model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        scheduler.step()

        # Validation
        if val_loader:
            tfm_model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for context_patches, padding_patches, x_future in val_loader:
                    context_patches = context_patches.to(device)
                    padding_patches = padding_patches.to(device).bool()
                    x_future = x_future.to(device)

                    output_tuple, _ = tfm_model(context_patches, padding_patches)
                    pred_raw = output_tuple[2]
                    B, num_patches, _ = pred_raw.shape
                    predictions = pred_raw.reshape(B, num_patches, o, q)
                    predictions_mean = predictions[..., 5]

                    target_len = x_future.shape[-1]
                    if target_len > o:
                        raise ValueError(f"horizon_len ({target_len}) must be <= output_patch_len ({o})")
                    last_patch_pred = predictions_mean[:, -1, :target_len]
                    val_loss += loss_fn(last_patch_pred, x_future).item()

            val_loss /= len(val_loader)

            print(f"Epoch {epoch+1}/{num_epochs} | Train: {train_loss:.6f} | Val: {val_loss:.6f}")
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(tfm_model.state_dict(), save_path)
                print(f"  -> Saved best model")
        else:
            print(f"Epoch {epoch+1}/{num_epochs} | Train: {train_loss:.6f}")

    return tfm_model


# Training
CONTEXT_LEN = 512
HORIZON_LEN = 64
PATCH_LEN = 32
BATCH_SIZE = 16
NUM_EPOCHS = 30
LR = 1e-5

torch.set_float32_matmul_precision("high")
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")

# Ensure patch_len matches model
PATCH_LEN = model.model.p

train_size = int(len(arr) * 0.9)
train_dataset = TimesFMDataset(arr[:train_size], CONTEXT_LEN, HORIZON_LEN, PATCH_LEN)
val_dataset = TimesFMDataset(arr[train_size:], CONTEXT_LEN, HORIZON_LEN, PATCH_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}")

finetuned = finetune_timesfm(
    model, train_loader, val_loader, NUM_EPOCHS, LR,
    save_path="./weights/timesfm/timesfm_full_finetuned/timesfm_finetuned.pth"
)
print("Complete Fine-tuning")



### Lag-Llama

https://colab.research.google.com/drive/1uvTmh-pe1zO5TeaaRVDdoEWJ5dFDI-pA?usp=sharing#scrollTo=00b11831-ea61-4dd8-a021-1cb10a1b4be6

https://github.com/time-series-foundation-models/lag-llama

https://www.ibm.com/think/tutorials/lag-llama

In [ ]:
# !pip uninstall gluonts -y
# !pip install "gluonts[torch]==0.14.4"

# !pip show pandas # 현재 버전 2.1.4
# !pip install pandas==2.0.3

# from huggingface_hub import hf_hub_download

# # 가중치 다운로드
# hf_hub_download(
#     repo_id="time-series-foundation-models/Lag-Llama",
#     filename="lag-llama.ckpt",
#     local_dir="../dependencies/lag-llama",
# )

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, '../dependencies/lag-llama')

import pandas as pd
import numpy as np
import torch
from gluonts.torch.distributions.studentT import StudentTOutput
from gluonts.torch.modules.loss import NegativeLogLikelihood
torch.serialization.add_safe_globals([StudentTOutput, NegativeLogLikelihood])
if not hasattr(StudentTOutput, "loss"):
    def loss(self, target, distr_args, loc=None, scale=None):
        distr = self.distribution(distr_args, loc=loc, scale=scale)
        return -distr.log_prob(target)
    StudentTOutput.loss = loss

from gluonts.dataset.common import ListDataset
from gluonts.evaluation import make_evaluation_predictions, Evaluator
from lag_llama.gluon.estimator import LagLlamaEstimator

from itertools import islice
from matplotlib import pyplot as plt
import matplotlib.dates as mdates
from tqdm.autonotebook import tqdm

from gluonts.dataset.repository.datasets import get_dataset
from gluonts.dataset.pandas import PandasDataset
from types import ModuleType

In [ ]:
# Lab Data
normal_df = pd.read_csv('../data/2st_collection/normal/normal_B-M-S1_imu_data_60Hz.csv', header=1)

features = [
    "accel_x", "accel_y", "accel_z",
    "gyro_x", "gyro_y", "gyro_z",
    "mag_x", "mag_y", "mag_z",
]

In [ ]:
# Ds Data
ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')

date = "2025-11-10"
ds_press_imu["timestamp"] = pd.to_datetime(ds_press_imu["timestamp"], utc=True, errors="coerce")
ds_press_imu = ds_press_imu.dropna(subset=["timestamp"])

mask = ds_press_imu["timestamp"].dt.date == pd.Timestamp(date).date()
normal_df = ds_press_imu[mask].reset_index(drop=True)

features = [
    "acc_x","acc_y","acc_z",
    "gyro_x","gyro_y","gyro_z",
    # "mag_x","mag_y","mag_z",
]

In [ ]:
arr = normal_df[features].to_numpy().astype(np.float32)  # (T, 9)

CKPT = "../dependencies/lag-llama/lag-llama.ckpt"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt = torch.load(CKPT, map_location=device, weights_only=False)

freq = "16ms" #64Hz
context_length = 512
prediction_length = 64
start = pd.Timestamp("2024-01-01")

T = arr.shape[0]
train_ratio = 0.9
split_idx = int(T * train_ratio)

# Train
train_ds = ListDataset([{
    "target": arr[:split_idx, i].T,
    "start": start,
        }
        for i in range(arr.shape[1])
    ], freq=freq)

# Validation
val_ds = ListDataset([{
    # "target": arr[:, i],
    "target": arr[:split_idx + prediction_length, i].T,
    "start": start,
        }
        for i in range(arr.shape[1])
    ], freq=freq
)

#마지막 prediction_length만 validation loss 계산에 사용
#내부적으로는 sliding window로 처리됨

In [ ]:
num_samples=100
estimator_args = ckpt["hyper_parameters"]["model_kwargs"]
lags_seq = ckpt["hyper_parameters"]["model_kwargs"]["lags_seq"]
# lags_seq = list(range(1, 85))

estimator = LagLlamaEstimator(
    ckpt_path=CKPT,
    prediction_length=prediction_length,
    context_length=context_length,
    aug_prob=0,
    lr=5e-4,
    rope_scaling={
        "type": "linear",
        "factor": max(1.0, (context_length + prediction_length) / estimator_args["context_length"]),
    },
    
    # device=torch.device('cpu'),
    # nonnegative_pred_samples=True,
    
    input_size=estimator_args["input_size"],
    n_layer=estimator_args["n_layer"],
    n_embd_per_head=estimator_args["n_embd_per_head"],
    n_head=estimator_args["n_head"],
    time_feat=estimator_args["time_feat"],
    
    lags_seq=lags_seq,
    
    batch_size=64,
    num_parallel_samples=num_samples,
    trainer_kwargs={"max_epochs": 50},
)

In [ ]:
# 학습 
predictor = estimator.train(
    training_data=train_ds,
    validation_data=val_ds,
    cache_data=True,
)
# 저장
save_dir = Path("./weights/lagllama/lagllama_ds_full_finetuned/")
save_dir.mkdir(parents=True, exist_ok=True)
predictor.serialize(save_dir)
print(f"저장 완료: {save_dir}")

### MOMENT

 https://github.com/moment-timeseries-foundation-model/moment

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.cuda.amp
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import OneCycleLR
from tqdm import tqdm

from momentfm import MOMENTPipeline
from momentfm.utils.utils import control_randomness
from momentfm.utils.forecasting_metrics import get_forecasting_metrics

control_randomness(seed=13)

In [ ]:
class MomentDataset(Dataset):
    """
    MOMENT 공식 InformerDataset과 동일한 형식
    """
    def __init__(
        self,
        data: np.ndarray,
        seq_len: int = 512,
        forecast_horizon: int = 64,
        stride: int = 64,
    ):
        self.seq_len = seq_len
        self.forecast_horizon = forecast_horizon
        self.n_channels = data.shape[1]
        
        self.samples = []
        total_len = seq_len + forecast_horizon
        
        for i in range(0, len(data) - total_len + 1, stride):
            self.samples.append(data[i:i + total_len])
        
        self.samples = np.array(self.samples, dtype=np.float32)
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        timeseries = sample[:self.seq_len].T.astype(np.float32)
        forecast = sample[self.seq_len:].T.astype(np.float32)
        input_mask = np.ones(self.seq_len, dtype=np.float32)
        
        return timeseries, forecast, input_mask

In [ ]:
# Lab Data
CSV_PATH = '../data/2st_collection/normal/normal_B-M-S1_imu_data_60Hz.csv'
df = pd.read_csv(CSV_PATH, header=1)

FEATURES = ["accel_x", "accel_y", "accel_z", "gyro_x", "gyro_y", "gyro_z", "mag_x", "mag_y", "mag_z"]

In [ ]:
# Ds Data
ds_press_imu = pd.read_csv('../data/ds_collection/ds_press_imu.csv')

date = "2025-11-10"
ds_press_imu["timestamp"] = pd.to_datetime(ds_press_imu["timestamp"], utc=True, errors="coerce")
ds_press_imu = ds_press_imu.dropna(subset=["timestamp"])

mask = ds_press_imu["timestamp"].dt.date == pd.Timestamp(date).date()
df = ds_press_imu[mask].reset_index(drop=True)

FEATURES = [
    "acc_x","acc_y","acc_z",
    "gyro_x","gyro_y","gyro_z",
    # "mag_x","mag_y","mag_z",
]

In [ ]:
SEQ_LEN = 512
FORECAST_HORIZON = 64
BATCH_SIZE = 16

model = MOMENTPipeline.from_pretrained(
    "AutonLab/MOMENT-1-large",
    model_kwargs={
        'task_name': 'forecasting',
        'forecast_horizon': FORECAST_HORIZON,
        'head_dropout': 0.1,
        'weight_decay': 0,
        'freeze_encoder': True,
        'freeze_embedder': True,
        'freeze_head': False, # head만 tuning
    },
)
model.init()

data = df[FEATURES].to_numpy().astype(np.float32)
print(f"Data shape: {data.shape}")

split_idx = int(len(data) * 0.9)
train_data, test_data = data[:split_idx], data[split_idx:]

train_dataset = MomentDataset(train_data, SEQ_LEN, FORECAST_HORIZON, stride=64)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_dataset = MomentDataset(test_data, SEQ_LEN, FORECAST_HORIZON, stride=64)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

cur_epoch = 0
MAX_EPOCH = 50

model = model.to(device)

criterion = criterion.to(device)
scaler = torch.cuda.amp.GradScaler()

max_lr = 1e-4
total_steps = len(train_loader) * MAX_EPOCH
scheduler = OneCycleLR(optimizer, max_lr=max_lr, total_steps=total_steps, pct_start=0.3)

# Gradient clipping value (공식 코드)
max_norm = 5.0

while cur_epoch < MAX_EPOCH:
    losses = []
    for timeseries, forecast, input_mask in tqdm(train_loader, total=len(train_loader)):
    # for timeseries, forecast, input_mask in tqdm(train_loader, desc=f"Epoch {cur_epoch}"):
        timeseries = timeseries.float().to(device)
        input_mask = input_mask.to(device)
        forecast = forecast.float().to(device)
        
        # timeseries, forecast, input_mask = next(iter(train_loader))
        # print("timeseries shape :", timeseries.shape)   # (B, C, L)
        # print("forecast shape   :", forecast.shape)     # (B, C, H)
        # print("input_mask shape :", input_mask.shape)   # (B, L)
        
        # Mixed precision training (공식 코드)
        with torch.cuda.amp.autocast():
            # takes in tensor of shape [batchsize, n_channels, context_length]
            output = model(x_enc=timeseries, input_mask=input_mask)
        loss = criterion(output.forecast, forecast)
        
        # Scales the loss, and calls backward() (공식 코드)
        scaler.scale(loss).backward()
        
        # Unscales gradients and clips them (공식 코드)
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        
        # Optimizer step (공식 코드)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        
        scheduler.step()
        losses.append(loss.item())
        
    losses = np.array(losses)
    avg_loss = np.mean(losses)
    print(f"Epoch {cur_epoch}: Train loss: {avg_loss:.3f}")
    
    cur_epoch += 1
    
    # Evaluation 
    trues, preds, histories, losses = [], [], [], []
    model.eval()
    with torch.no_grad():
        for timeseries, forecast, input_mask in tqdm(test_loader, total=len(test_loader)):
        # for timeseries, forecast, input_mask in tqdm(test_loader, desc="Testing"):
            timeseries = timeseries.float().to(device)
            input_mask = input_mask.to(device)
            forecast = forecast.float().to(device)
            
            with torch.cuda.amp.autocast():
                output = model(x_enc=timeseries, input_mask=input_mask)
            loss = criterion(output.forecast, forecast)   
                         
            losses.append(loss.item())
            trues.append(forecast.detach().cpu().numpy())
            preds.append(output.forecast.detach().cpu().numpy())
            histories.append(timeseries.detach().cpu().numpy())
    
    losses = np.array(losses)
    average_loss = np.average(losses)
    model.train()
    
    trues = np.concatenate(trues, axis=0)
    preds = np.concatenate(preds, axis=0)
    histories = np.concatenate(histories, axis=0)
    
    metrics = get_forecasting_metrics(y=trues, y_hat=preds, reduction='mean')
    print(f"Epoch {cur_epoch}: Test MSE: {metrics.mse:.3f} | Test MAE: {metrics.mae:.3f}")

# 저장

save_dir = Path("./weights/moment/moment_full_finetuned/")
save_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), "./weights/moment/moment_full_finetuned/moment_finetuned.pth")
print("Model saved to ./weights/moment/moment_full_finetuned/moment_finetuned.pth")
